# 07 · El juicio · antes vs después

**Parte III · El oficio.** Entrenaste (o vas a entrenar) los lentes. ¿Sirvieron? "Se ve mejor" no es respuesta. Este notebook arma la tabla que pide el reto: **mismos 5 prompts de hold-out**, varias condiciones, un juez reproducible y una columna para tu juicio humano.

Condiciones:

| Condición | Qué es | ¿Entrena? |
|---|---|---|
| `zero-shot` | Qwen Instruct + system de Don Titular + notas | no |
| `few-shot` | lo mismo + **un** ejemplo resuelto del dataset antes de las notas | no |
| `lora` | tu adapter de `sft_lora_noticias.py` | sí |

Las dos primeras ya vienen guardadas (`salidas_referencia.json`: corrida real de `Qwen2.5-0.5B-Instruct`, greedy). La tercera la generas tú con `GENERAR = True` y tu adapter. Sin torch, el notebook corre igual con lo guardado.

## Dummy (antes de la fórmula)

Un examen justo tiene tres reglas: preguntas que el alumno **no vio** (hold-out), la misma rúbrica para todos (el juez) y alguien que revise lo que la rúbrica no ve (tú).

Si la tabla dice que el LoRA mejora pero tú no publicarías ninguno de sus briefings, la tabla está midiendo otra cosa.

## Matemática (la misma idea, en serio)

Para cada condición $c$ y prompt $i\in\{1,\dots,n\}$ el juez da tres indicadores: $F_{ci}$ (formato), $C_{ci}$ (citas) y $S_{ci}\in[0,1]$ (soporte). Reportamos medias, por ejemplo

$$
\bar F_c = \frac1n\sum_i F_{ci}, \qquad \bar S_c = \frac{1}{|I_c|}\sum_{i\in I_c} S_{ci},
$$

con $I_c$ los prompts donde hubo algo que medir. Con $n=5$ cada prompt pesa 20%: una diferencia de un caso **no** es una conclusión. El reto pide la tabla para mostrar el patrón, no un p-valor.

**Ley de Goodhart.** Si entrenas para maximizar al juez, el modelo aprende a copiar títulos (soporte 100%, formato perfecto, cero resumen). Por eso la columna humana no es decoración.

## 0 · Cargar hold-out, referencia y juez

In [1]:
# Colab: descomenta (una vez).
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/parte-3-oficio/don_titular.py
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/parte-3-oficio/holdout.jsonl
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/parte-3-oficio/salidas_referencia.json
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/parte-3-oficio/ejemplo-briefing.jsonl

import json
import sys
from pathlib import Path

OFICIO = next((p for p in (Path("."), Path("../parte-3-oficio")) if (p / "don_titular.py").exists()), Path("."))
sys.path.insert(0, str(OFICIO.resolve()))
import don_titular as dt

HOLDOUT = [json.loads(linea) for linea in (OFICIO / "holdout.jsonl").read_text(encoding="utf-8").splitlines()]
REF = json.loads((OFICIO / "salidas_referencia.json").read_text(encoding="utf-8"))
SALIDAS: dict[str, dict[str, str]] = {k: dict(v) for k, v in REF["salidas"].items()}

print("referencia:", REF["meta"])
print("hold-out:", [c["id"] for c in HOLDOUT])

referencia: {'modelo': 'Qwen/Qwen2.5-0.5B-Instruct', 'decodificacion': 'greedy', 'max_new_tokens': 220, 'fecha': '2026-10-02'}
hold-out: ['h1-economia', 'h2-clima', 'h3-deportes', 'h4-movilidad', 'h5-unam']


## 1 · (Opcional) Generar: tu adapter

Pon `GENERAR = True` en Colab con GPU, después de correr el gym. Si `ADAPTER` existe, agrega la columna `lora`. Si además pones `REGENERAR_REFERENCIA = True`, vuelve a generar zero-shot y few-shot con tu entorno (útil para comparar en la misma máquina).

In [2]:
GENERAR = False
REGENERAR_REFERENCIA = False
MODELO = REF["meta"]["modelo"]
ADAPTER = OFICIO / "don-titular-lora"

if GENERAR:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(MODELO)
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    modelo = AutoModelForCausalLM.from_pretrained(MODELO).to(dispositivo).eval()
    ejemplo = json.loads((OFICIO / "ejemplo-briefing.jsonl").read_text(encoding="utf-8").splitlines()[0])["messages"]

    def generar(m, mensajes):
        entrada = tok.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(dispositivo)
        with torch.no_grad():
            out = m.generate(**entrada, max_new_tokens=REF["meta"]["max_new_tokens"], do_sample=False,
                             temperature=None, top_p=None, top_k=None, pad_token_id=tok.eos_token_id)
        return tok.decode(out[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    if REGENERAR_REFERENCIA:
        for caso in HOLDOUT:
            SALIDAS[caso["id"]]["zero-shot"] = generar(modelo, dt.mensajes(caso["notas"]))
            SALIDAS[caso["id"]]["few-shot"] = generar(modelo, ejemplo + [{"role": "user", "content": caso["notas"]}])

    if ADAPTER.exists():
        from peft import PeftModel

        con_lentes = PeftModel.from_pretrained(modelo, str(ADAPTER)).eval()
        for caso in HOLDOUT:
            SALIDAS[caso["id"]]["lora"] = generar(con_lentes, dt.mensajes(caso["notas"]))
    else:
        print(f"No hay adapter en {ADAPTER}: corre sft_lora_noticias.py primero.")

CONDICIONES = [c for c in ("zero-shot", "few-shot", "lora") if all(c in SALIDAS[k] for k in SALIDAS)]
print("condiciones:", CONDICIONES)

condiciones: ['zero-shot', 'few-shot']


## 2 · La tabla

In [3]:
JUICIOS = {(caso["id"], c): dt.juzgar(caso["notas"], SALIDAS[caso["id"]][c]) for caso in HOLDOUT for c in CONDICIONES}


def celda(j: dt.Juicio) -> str:
    s = "—" if j.soporte is None else f"{j.soporte:.0%}"
    return f"F{'✔' if j.formato_ok else '✘'} C{'✔' if j.citas_ok else '✘'} S{s:>4}"


print(f"{'prompt':14}" + "".join(f"{c:>18}" for c in CONDICIONES))
for caso in HOLDOUT:
    print(f"{caso['id']:14}" + "".join(f"{celda(JUICIOS[caso['id'], c]):>18}" for c in CONDICIONES))

print("\nF = 5 bullets numerados · C = cada hecho cita un medio de las notas · S = soporte en las notas\n")
print(f"{'resumen':14}" + "".join(f"{c:>18}" for c in CONDICIONES))
for nombre, f in (
    ("formato", lambda js: f"{sum(j.formato_ok for j in js) / len(js):.0%}"),
    ("citas", lambda js: f"{sum(j.citas_ok for j in js) / len(js):.0%}"),
    ("soporte medio", lambda js: f"{sum(s) / len(s):.0%}" if (s := [j.soporte for j in js if j.soporte is not None]) else "—"),
    ("problemas", lambda js: str(sum(len(j.problemas) for j in js))),
):
    print(f"{nombre:14}" + "".join(f"{f([JUICIOS[k, c] for k in SALIDAS]):>18}" for c in CONDICIONES))

prompt                 zero-shot          few-shot
h1-economia          F✘ C✘ S100%       F✘ C✘ S100%
h2-clima             F✘ C✔ S100%       F✘ C✔ S100%
h3-deportes          F✘ C✘ S 75%       F✘ C✔ S100%
h4-movilidad         F✘ C✘ S100%       F✘ C✘ S100%
h5-unam              F✘ C✔ S100%       F✘ C✘ S100%

F = 5 bullets numerados · C = cada hecho cita un medio de las notas · S = soporte en las notas

resumen                zero-shot          few-shot
formato                       0%                0%
citas                        40%               40%
soporte medio                95%              100%
problemas                     14                14


**Cómo leer la tabla (corrida guardada).**

- **Formato 0% en las dos.** Ninguna da los 5 bullets con "no aparece" de relleno. Qwen 0.5B entiende "resume" pero no "exactamente 5, y rellena". Esa regla de oficio es justo lo que un LoRA aprende de 100 ejemplos y un prompt no logra.
- **Few-shot cambia la forma, no tanto el número.** Con un ejemplo enfrente responde en renglones `1) …` cortos y adorna menos (soporte 100% vs 95%: **un** caso de cinco, no una conclusión). Un ejemplo en el prompt ya es "entrenar" sin gradiente (*in-context learning*); el costo es que viaja en **cada** llamada.
- **Las citas fallan distinto.** Zero-shot cita en markdown (`**ESPN**:`) y a veces inventa el medio; few-shot copia `1) El Economista:` sin paréntesis. El juez marca las dos.

Tu adapter tiene que ganarle a **few-shot**, no solo a zero-shot. Si no le gana, el LoRA no se justificó: un prompt con un ejemplo era más barato.

## 3 · Lo que el juez no vio

Lee las salidas. Estos dos casos están en la corrida guardada.

In [4]:
def mostrar(caso_id: str, condicion: str) -> None:
    notas = next(c["notas"] for c in HOLDOUT if c["id"] == caso_id)
    j = JUICIOS[caso_id, condicion]
    print(f"══ {caso_id} · {condicion} · {j.resumen()}\n── notas\n{notas}\n── salida\n{SALIDAS[caso_id][condicion]}\n")


mostrar("h1-economia", "zero-shot")
mostrar("h4-movilidad", "few-shot")

══ h1-economia · zero-shot · formato ✘ · citas ✘ · soporte 100% · 4 problema(s)
── notas
Notas del 2026-08-20:
1. El Economista — El peso cierra la semana en 18.40 por dólar.
   Analistas lo atribuyen a la expectativa de recortes de la Fed.
2. Expansión — Pemex reporta pérdida trimestral menor a la esperada.
3. El País — La inflación de la primera quincena sube a 4.1 %.
── salida
**Resumen:**
- **Economía**: El peso cerró la semana en 18.40 dólares por dólar, con la expectativa de recortes de la Fed.
- **Expansión**: Pemex reportó una pérdida trimestral menor a la esperada.
- **La República**: La inflación de la primera quincena subió a 4.1%, lo que es menos que la prevista.

**Medio:** El Economista, Expansión, El País

══ h4-movilidad · few-shot · formato ✘ · citas ✘ · soporte 100% · 5 problema(s)
── notas
Notas del 2026-08-23:
1. Local — Cierre de Reforma por marcha a partir de las 10:00.
   Se recomiendan Insurgentes y Circuito Interior como alternativas.
2. Metro CDMX — Marcha len

- **h1 · zero-shot.** "…subió a 4.1%, lo que es menos que la prevista": las notas no dicen nada de lo previsto. De las 6 raíces del renglón, 3 están en las notas (`infla`, `prime`, `quinc`): soporte exactamente 50%, el umbral, y pasa. El juez sí cachó el medio inventado ("La República") y "Economía" usado como si fuera un medio.
- **h4 · few-shot.** Soporte 100%, y aun así está mal: le atribuye el cierre por la marcha a **Metro CDMX** (era *Local*) y convierte la avenida **Reforma** en un medio. Las palabras están en las notas; las relaciones no. Un juez de bolsa de palabras no ve relaciones.

## 4 · Tu columna: ¿lo publicarías?

Llena 1 (sí, tal cual), 0.5 (con una corrección) o 0 (no). Es la columna que el reto toma en serio.

In [5]:
HUMANO: dict[tuple[str, str], float | None] = {(k, c): None for k in SALIDAS for c in CONDICIONES}
# Ejemplo: HUMANO["h4-movilidad", "few-shot"] = 0   # atribuye el cierre al Metro

for c in CONDICIONES:
    notas = [v for (k, cc), v in HUMANO.items() if cc == c and v is not None]
    print(f"{c:10} · humano: " + (f"{sum(notas) / len(notas):.2f} ({len(notas)}/{len(SALIDAS)} revisados)" if notas else "sin revisar"))

zero-shot  · humano: sin revisar
few-shot   · humano: sin revisar


---

## Entregable (reto)

1. La tabla de la sección 2 con **tres** columnas (`zero-shot`, `few-shot`, `lora`) en **tu** hold-out de ≥ 5 prompts, separado por fecha (notebook 06).
2. La columna humana llena, y un caso donde el juez y tú no estén de acuerdo.
3. Una frase honesta: ¿tu LoRA le gana a few-shot? Si no, ¿por qué crees (datos, épocas, r)?

## Para romperlo

- Agrega a `don_titular.juzgar` una regla de **atribución**: el medio citado en un bullet debe ser el de la nota con mayor soporte para ese bullet. ¿Cuántos casos nuevos caza en h4?
- Haz few-shot con **tres** ejemplos. ¿Se acerca al formato? ¿Cuántos tokens extra cuesta por llamada?
- Entrena un LoRA con un dataset donde **todas** las filas copien títulos. Mira cómo sube el juez y baja tu columna. Eso es Goodhart.